# Ronchi-ML (Colab, with Google Drive)

Baseline: **p_corr** with **auxiliary offset** (multitask), OneCycleLR, optional offset reporting at inference.

This version mounts Google Drive so you can **load your dataset** (manifest + images) from Drive and **save models** back to Drive.


In [ ]:
import sys, subprocess
def pip_install(pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)

pip_install(['torch', 'torchvision', 'opencv-python', 'numpy'])
print('✅ Deps installed')

## Mount Google Drive and set paths
Edit the two directory variables below to point to your dataset and where you want models saved. The dataset directory should contain `manifest.jsonl` and an `images/` subfolder (or adjust the paths accordingly).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# === EDIT THESE ===
DRIVE_DATA_DIR = '/content/drive/MyDrive/ronchi-ml/data'    # contains manifest.jsonl and images/
DRIVE_MODELS_DIR = '/content/drive/MyDrive/ronchi-ml/models' # where checkpoints will be saved

print('DATA from :', DRIVE_DATA_DIR)
print('MODELS to :', DRIVE_MODELS_DIR)

## Runtime project files
Writes `src/` code (data.py, model.py, train.py, infer.py) into the Colab runtime.

In [ ]:
import os, textwrap, json
from pathlib import Path
root = Path('.')
src = root / 'src'
src.mkdir(parents=True, exist_ok=True)

data_py = r'''from dataclasses import dataclass
import json, cv2 as cv, numpy as np, torch
from torch.utils.data import Dataset

@dataclass
class CondNorm:
    f_mu: float
    f_sigma: float
    lpi_mu: float
    lpi_sigma: float

@dataclass
class OffsetNorm:
    off_mu: float
    off_sigma: float

class RonchiJSONLDataset(Dataset):
    """JSONL format:
    { "image": "...png",
      "meta": {"f":..., "lpi":..., "offset": ...},
      "labels": {"p_corr": ...}
    }
    - p_corr is used in natural units [0..1] (no label normalization).
    - offset is normalized using train-split stats: (offset - mu)/sigma.
    """
    def __init__(self, jsonl_path, cond_norm: CondNorm, off_norm: OffsetNorm, resize=320, augment=False):
        self.items = []
        with open(jsonl_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                self.items.append(json.loads(line))
        self.C = cond_norm
        self.O = off_norm
        self.resize = int(resize)
        self.augment = augment

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        it = self.items[idx]
        img = cv.imread(it['image'], cv.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(it['image'])
        img = cv.resize(img, (self.resize, self.resize), interpolation=cv.INTER_AREA)
        img = img.astype(np.float32) / 255.0
        img = (img - 0.5) / 0.5

        # Optional phase jitter (disabled by default; set augment=True)
        if self.augment:
            max_px = max(1, self.resize // 64)  # ~5 px at 320
            d = int(np.random.randint(-max_px, max_px+1))
            if d != 0:
                img = np.roll(img, shift=d, axis=0)  # vertical shift ~ stripe-normal

        x = torch.from_numpy(img[None, ...])  # [1,H,W]

        f   = float(it['meta']['f']); lpi = float(it['meta']['lpi'])
        off = float(it['meta']['offset'])  # inches

        # Normalize cond
        f_n   = (f   - self.C.f_mu)   / (self.C.f_sigma   + 1e-6)
        lpi_n = (lpi - self.C.lpi_mu) / (self.C.lpi_sigma + 1e-6)
        cond = torch.tensor([f_n, lpi_n], dtype=torch.float32)

        # Targets
        p_corr = float(it['labels']['p_corr'])                  # natural [0,1]
        y_p = torch.tensor([p_corr], dtype=torch.float32)
        y_off = torch.tensor([(off - self.O.off_mu)/(self.O.off_sigma + 1e-6)], dtype=torch.float32)

        return x, cond, y_p, y_off
'''

model_py = r'''import torch, torch.nn as nn

class Backbone(nn.Module):
    def __init__(self, C=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, C, 5, 2, 2), nn.ReLU(inplace=True),
            nn.Conv2d(C, C, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(C, 2*C, 3, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(2*C, 2*C, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(2*C, 4*C, 3, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(4*C, 4*C, 3, 1, 1), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4,4))  # preserve a 4x4 grid of spatial cues
        )
        self.out_dim = 4*4*4*C

    def forward(self, x):
        return self.features(x).flatten(1)  # [B, D]

class Net(nn.Module):
    def __init__(self, cond_dim=2):
        super().__init__()
        self.backbone = Backbone(C=64)
        D = self.backbone.out_dim
        self.cond_proj = nn.Sequential(nn.Linear(cond_dim, 32), nn.ReLU(inplace=True))
        self.fuse = nn.Sequential(nn.Linear(D + 32, 256), nn.ReLU(inplace=True))
        self.head_p = nn.Sequential(nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Linear(128, 1))
        self.head_off = nn.Sequential(nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Linear(128, 1))

    def forward(self, img, cond):
        h_img = self.backbone(img)
        h_c   = self.cond_proj(cond)
        h     = torch.cat([h_img, h_c], dim=1)
        h     = self.fuse(h)
        p     = self.head_p(h)
        off_n = self.head_off(h)
        return p, off_n
'''

train_py = r'''import argparse, json, numpy as np, torch, torch.nn as nn, tempfile, os, random, math
from torch.utils.data import DataLoader
from data import RonchiJSONLDataset, CondNorm, OffsetNorm
from model import Net

def _ensure_parent(path):
    d = os.path.dirname(path)
    if d and not os.path.exists(d):
        os.makedirs(d, exist_ok=True)

def _scan_stats(jsonl_path):
    P, F, L, OFF = [], [], [], []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            o = json.loads(line)
            P.append(float(o['labels']['p_corr']))
            F.append(float(o['meta']['f']))
            L.append(float(o['meta']['lpi']))
            OFF.append(float(o['meta']['offset']))
    def stats(a):
        return float(np.mean(a)), float(np.std(a) + 1e-6)
    (f_mu, f_sd)   = stats(F)
    (l_mu, l_sd)   = stats(L)
    (off_mu, off_sd) = stats(OFF)
    Cnorm = CondNorm(f_mu=f_mu, f_sigma=f_sd, lpi_mu=l_mu, lpi_sigma=l_sd)
    Onorm = OffsetNorm(off_mu=off_mu, off_sigma=off_sd)
    return Cnorm, Onorm

def _split_manifest_inline(manifest_path, val_ratio=0.2, seed=42):
    with open(manifest_path) as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    rng = random.Random(seed)
    rng.shuffle(lines)
    cut = int((1.0 - val_ratio) * len(lines))
    train_lines, val_lines = lines[:cut], lines[cut:]
    tf_tr = tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False, prefix='_tmp_train_', dir='.')
    tf_va = tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False, prefix='_tmp_val_', dir='.')
    tf_tr.write("\n".join(train_lines) + "\n"); tf_tr.flush(); tf_tr.close()
    tf_va.write("\n".join(val_lines) + "\n"); tf_va.flush(); tf_va.close()
    return tf_tr.name, tf_va.name

def _make_scheduler(opt, dl_tr_len, args):
    if args.sched == "onecycle":
        from torch.optim.lr_scheduler import OneCycleLR
        steps_per_epoch = max(1, dl_tr_len)
        sched = OneCycleLR(
            opt,
            max_lr=args.lr,
            epochs=args.epochs,
            steps_per_epoch=steps_per_epoch,
            pct_start=args.pct_start,
            div_factor=args.div_factor,
            final_div_factor=args.final_div_factor,
            three_phase=False,
            anneal_strategy="cos"
        )
        return sched, "batch"
    else:
        return None, None

def train(args):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    cleanup_paths = []
    if args.manifest:
        tr_path, va_path = _split_manifest_inline(args.manifest, args.val_ratio, args.split_seed)
        cleanup_paths.extend([tr_path, va_path])
    else:
        tr_path, va_path = args.train_jsonl, args.val_jsonl

    Cnorm, Onorm = _scan_stats(tr_path)

    # Ensure output dirs exist
    _ensure_parent(args.norm_cond)
    _ensure_parent(args.norm_offset)
    _ensure_parent(args.out_ckpt)

    with open(args.norm_cond, 'w') as f: json.dump(Cnorm.__dict__, f, indent=2)
    with open(args.norm_offset, 'w') as f: json.dump(Onorm.__dict__, f, indent=2)

    ds_tr = RonchiJSONLDataset(tr_path, Cnorm, Onorm, resize=args.resize, augment=args.augment)
    ds_va = RonchiJSONLDataset(va_path, Cnorm, Onorm, resize=args.resize, augment=False)
    dl_tr = DataLoader(ds_tr, batch_size=args.bs, shuffle=True, num_workers=0, pin_memory=False)
    dl_va = DataLoader(ds_va, batch_size=args.bs, shuffle=False, num_workers=0, pin_memory=False)

    model = Net().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.wd)
    sched, sched_mode = _make_scheduler(opt, len(dl_tr), args)
    l1 = nn.SmoothL1Loss()

    best = float('inf')
    try:
        for ep in range(args.epochs):
            model.train()
            running = {"loss":0.0, "p":0.0, "off":0.0, "n":0}
            for img, cond, y_p, y_off in dl_tr:
                img, cond, y_p, y_off = img.to(device), cond.to(device), y_p.to(device), y_off.to(device)
                p_hat, off_hat = model(img, cond)
                loss_p   = l1(p_hat, y_p)                 # p_corr in natural units
                loss_off = l1(off_hat, y_off)             # offset in normalized units
                loss = loss_p + args.off_weight * loss_off

                opt.zero_grad(); loss.backward()
                if args.clip > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=args.clip)
                opt.step()
                if sched and sched_mode == "batch":
                    sched.step()

                # accum
                bs = img.size(0)
                running["loss"] += loss.item() * bs
                running["p"]    += loss_p.item() * bs
                running["off"]  += loss_off.item() * bs
                running["n"]    += bs

            # Epoch summary
            denom = max(running["n"],1)
            avg_loss = running["loss"]/denom
            avg_p    = running["p"]/denom
            avg_off  = running["off"]/denom

            # Validation (report p_corr MAE in natural units, clamped to [0,1], and offset MAE in inches)
            model.eval()
            n, p_mae, off_mae_in = 0, 0.0, 0.0
            with torch.no_grad():
                for img, cond, y_p, y_off in dl_va:
                    img, cond, y_p, y_off = img.to(device), cond.to(device), y_p.to(device), y_off.to(device)
                    p_hat, off_hat = model(img, cond)
                    # p_corr MAE (clamped for readability)
                    p_mae += torch.abs(torch.clamp(p_hat, 0, 1) - y_p).sum().item()
                    # offset MAE in inches (denormalize)
                    off_pred_in = off_hat * Onorm.off_sigma + Onorm.off_mu
                    off_true_in = y_off * Onorm.off_sigma + Onorm.off_mu
                    off_mae_in += torch.abs(off_pred_in - off_true_in).sum().item()
                    n += img.size(0)
            p_mae /= max(n,1)
            off_mae_in /= max(n,1)

            print(f"Epoch {ep+1}/{args.epochs}  train| loss={avg_loss:.4f} p_loss={avg_p:.4f} off_loss={avg_off:.4f}  "
                  f"val| p_corr_MAE={p_mae:.4f}  offset_in_MAE={off_mae_in:.3f}\"  lr={opt.param_groups[0]['lr']:.2e}")

            if p_mae < best:
                best = p_mae
                # Ensure models directory exists in runtime and Drive
                os.makedirs('models', exist_ok=True)
                os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
                # Save to runtime
                torch.save({
                    "model": model.state_dict(),
                    "cond_norm": Cnorm.__dict__,
                    "offset_norm": Onorm.__dict__
                }, args.out_ckpt)
                print(f"✔ saved runtime {args.out_ckpt}")
                # Also copy to Drive (mirror filename)
                import shutil
                shutil.copy2(args.out_ckpt, os.path.join(DRIVE_MODELS_DIR, os.path.basename(args.out_ckpt)))
                print(f"⬆️ copied to Drive: {os.path.join(DRIVE_MODELS_DIR, os.path.basename(args.out_ckpt))}")
    finally:
        for p in cleanup_paths:
            try: os.remove(p)
            except Exception: pass

if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    mode = ap.add_mutually_exclusive_group(required=True)
    mode.add_argument('--manifest', help='Single JSONL file; will be split into train/val internally.')
    mode.add_argument('--train-jsonl', help='Training JSONL (if not using --manifest).')
    ap.add_argument('--val-jsonl', help='Validation JSONL (required if using --train-jsonl).')

    ap.add_argument('--out-ckpt',       default='models/ronchi_auxoffset.pt')
    ap.add_argument('--norm-cond',      default='models/cond_norm.json')
    ap.add_argument('--norm-offset',    default='models/offset_norm.json')

    ap.add_argument('--resize', type=int, default=320)
    ap.add_argument('--bs',     type=int, default=8)
    ap.add_argument('--lr',     type=float, default=3e-3)
    ap.add_argument('--epochs', type=int, default=80)
    ap.add_argument('--wd',     type=float, default=0.0, help='Weight decay (Adam)')
    ap.add_argument('--clip',   type=float, default=0.0, help='Gradient clip max-norm (0 disables)')

    ap.add_argument('--sched',  choices=['onecycle','none'], default='onecycle')
    ap.add_argument('--pct-start', type=float, default=0.3, help='OneCycleLR warmup fraction')
    ap.add_argument('--div-factor', type=float, default=25.0, help='OneCycleLR initial_lr = max_lr/div_factor')
    ap.add_argument('--final-div-factor', type=float, default=1e3, help='OneCycleLR min_lr = initial_lr/final_div_factor')

    ap.add_argument('--off-weight', type=float, default=0.75, help='Aux offset loss weight')
    ap.add_argument('--augment', action='store_true', help='Enable phase-jitter augmentation during training.')

    ap.add_argument('--val-ratio', type=float, default=0.2, help='Validation fraction (default 0.2).')
    ap.add_argument('--split-seed', type=int, default=42, help='Shuffle seed for reproducibility.')

    args = ap.parse_args()
    if args.train_jsonl and not args.val_jsonl:
        ap.error('--val-jsonl is required when --train-jsonl is used.')

    train(args)
'''

infer_py = r'''import argparse, json, torch, numpy as np, cv2 as cv, os, shutil
from data import CondNorm, OffsetNorm
from model import Net

def _prep_image(path, resize=320):
    img = cv.imread(path, cv.IMREAD_GRAYSCALE)
    if img is None: raise FileNotFoundError(path)
    img = cv.resize(img, (resize, resize), interpolation=cv.INTER_AREA)
    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    return torch.from_numpy(img[None, None, ...])

def _prep_cond(f, lpi, C: CondNorm):
    f_n   = (f   - C.f_mu)   / (C.f_sigma   + 1e-6)
    lpi_n = (lpi - C.lpi_mu) / (C.lpi_sigma + 1e-6)
    return torch.tensor([[f_n, lpi_n]], dtype=torch.float32)

def infer(image, f, lpi, ckpt, resize=320, report_offset=False, drive_models_dir=None):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    ck = torch.load(ckpt, map_location=device)
    C = CondNorm(**ck['cond_norm'])
    O = OffsetNorm(**ck['offset_norm'])

    model = Net().to(device)
    model.load_state_dict(ck['model'])
    model.eval()

    x = _prep_image(image, resize).to(device)
    cond = _prep_cond(f, lpi, C).to(device)

    out = {}
    with torch.no_grad():
        p, off_n = model(x, cond)
        p_corr = torch.clamp(p, 0, 1).squeeze().item()
        out['p_corr'] = round(float(p_corr), 4)
        out['notes'] = 'aux-offset trained' if report_offset else 'aux-offset trained, offset ignored at inference'
        if report_offset:
            off_in = (off_n.squeeze() * O.off_sigma + O.off_mu).item()
            out['offset_in'] = round(float(off_in), 3)

    # optionally copy checkpoint to Drive location (if not already)
    if drive_models_dir:
        os.makedirs(drive_models_dir, exist_ok=True)
        dst = os.path.join(drive_models_dir, os.path.basename(ckpt))
        if not os.path.exists(dst):
            shutil.copy2(ckpt, dst)
    return out

if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--image', required=True)
    ap.add_argument('--f', type=float, required=True)
    ap.add_argument('--lpi', type=float, required=True)
    ap.add_argument('--ckpt', default='models/ronchi_auxoffset.pt')
    ap.add_argument('--resize', type=int, default=320)
    ap.add_argument('--report-offset', action='store_true', help='Also output estimated offset (inches).')
    ap.add_argument('--drive-models-dir', default=None, help='If set, copy the checkpoint to this Drive folder.')
    args = ap.parse_args()
    res = infer(args.image, args.f, args.lpi, args.ckpt, resize=args.resize, report_offset=args.report_offset, drive_models_dir=args.drive_models_dir)
    print(json.dumps(res, indent=2))
'''

(src / 'data.py').write_text(data_py)
(src / 'model.py').write_text(model_py)
(src / 'train.py').write_text(train_py)
(src / 'infer.py').write_text(infer_py)
print('✅ Wrote src files:', [p.name for p in src.glob('*.py')])

## Copy dataset from Drive to runtime
Copies `manifest.jsonl` and `images/` from Drive to local runtime paths expected by the code.

> If your layout differs, edit the `cp` commands accordingly.

In [ ]:
import os
os.makedirs('data/images', exist_ok=True)

# Copy manifest and images from Drive to runtime
!cp "$DRIVE_DATA_DIR/manifest.jsonl" ./manifest.jsonl
!cp -r "$DRIVE_DATA_DIR/images" ./data/
!echo 'Copied manifest and images to runtime.'

# (Optional) peek at first few lines
!head -n 3 manifest.jsonl || true

## Tiny sanity checks (optional)

In [ ]:
import os
os.makedirs('models', exist_ok=True)

# 1-sample overfit
!head -n 1 manifest.jsonl > tiny1.jsonl
!python src/train.py --train-jsonl tiny1.jsonl --val-jsonl tiny1.jsonl \
  --resize 320 --epochs 120 --lr 0.01 --bs 1 --off-weight 0.5 --sched onecycle

# 8-sample
# !head -n 8 manifest.jsonl > tiny8.jsonl
# !python src/train.py --train-jsonl tiny8.jsonl --val-jsonl tiny8.jsonl \
#   --resize 320 --epochs 120 --lr 0.003 --bs 4 --off-weight 0.5 --sched onecycle

## Full training (auto split 80/20)
Checkpoints are saved to runtime **and** copied to Drive (`DRIVE_MODELS_DIR`).

In [ ]:
!python src/train.py --manifest manifest.jsonl \
  --resize 320 --epochs 80 --lr 0.003 --bs 8 \
  --off-weight 0.75 --sched onecycle

## Inference
You can load the model from runtime (./models) or directly from your Drive copy. The command below uses the runtime path and also ensures the checkpoint is mirrored to Drive.

In [ ]:
# Example: adjust image path and parameters
!python src/infer.py --image data/images/ronchi_3.0_-0.5_0.0.png \
  --f 3.0 --lpi 100 --resize 320 --report-offset \
  --ckpt models/ronchi_auxoffset.pt \
  --drive-models-dir "$DRIVE_MODELS_DIR"

## Notes
- Keep `--resize` consistent between training and inference.
- If convergence is slow on your dataset, try smaller batch sizes (e.g., `--bs 4`) or increase `--off-weight` to `0.75–1.0`.
- For robustness to phase/offset jitter, add `--augment` during training once the loss is trending down.